In [1]:
import pandas as pd
import duckdb
from pathlib import Path

### ¿Qué haremos exactamente en esta etapa y por qué es vital?

En lugar de crear bases de datos físicas todavía, vamos a usar el "superpoder" principal de DuckDB: su motor en memoria. DuckDB puede leer tu DataFrame de Pandas (o tu archivo CSV de la capa Silver) directamente en la RAM usando SQL puro, sin necesidad de comandos CREATE TABLE.

El objetivo de esta fase es el QA (Quality Assurance):
Antes de modelar dimensiones y hechos (Fact/Dimensions tables), debemos validar que la limpieza de Pandas fue exitosa verificando las métricas del negocio.

Responderemos preguntas de validación como:

¿Hay algún Total Spent que sea negativo o cero por error?

¿Coincide el total de ventas facturadas (Total Revenue) con lo que esperaba el negocio?

¿Tenemos transacciones huérfanas o fechas fuera del rango esperado (ej: años 2099)?

In [2]:
# Ruta del clean_cafe_sales.csv
project_root : Path = Path.cwd().parent
silver_csv_path : Path = project_root / "data" / "silver" / "clean_cafe_sales.csv"

# Cargar en memoria
df_silver: pd.DataFrame = pd.read_csv(silver_csv_path)

In [3]:
# Ejecución de DuckDB

query_validation = """
    SELECT
        COUNT(*) as total_filas,
        COUNT(DISTINCT transaction_id) as transacciones_unicas,
        MIN(transaction_date) as primera_fecha,
        MAX(transaction_date) as ultima_fecha,
    FROM df_silver
"""

# duckdb.query() ejecuta el SQL sobre el DataFrame de Pandas 'df_silver'
df_results : pd.DataFrame = duckdb.query(query_validation).df()

df_results

,total_filas,transacciones_unicas,primera_fecha,ultima_fecha
0,9067,9067,2023-01-01,2023-12-31


Sin mover datos, sin crear bases físicas, estamos usando SQL analítico para auditar la salud de los datos que limpiaste. Si total_filas es igual a transacciones_unicas. Lo cual es correcto

---
### (Integridad de Datos)
En el mundo real, los sistemas de origen (como las cajas registradoras) tienen errores. A veces la cantidad es negativa, o el total no coincide con la multiplicación de la cantidad por el precio. Vamos a validar que tu capa Silver sea matemáticamente perfecta.

In [4]:
query_validate_integrity = """
    SELECT 
        COUNT(*) as tickets_con_error,
        SUM(CASE WHEN quantity <= 0 THEN 1 ELSE 0 END) as error_cantidad,
        SUM(CASE WHEN price_per_unit < 0 THEN 1 ELSE 0 END) as error_price,
        SUM(
            CASE 
                WHEN ROUND(total_spent, 2) != ROUND((quantity * price_per_unit), 2) THEN 1
                ELSE 0
            END 
        ) as error_calculo_total
    FROM df_silver
"""

df_data_integrity : pd.DataFrame = duckdb.query(query_validate_integrity).df()

df_data_integrity

,tickets_con_error,error_cantidad,error_price,error_calculo_total
0,9067,0.0,0.0,0.0


Esto confirmaría que no tenemos datos erroneos y realizamos una buena limpieza.

---
### SQL Analytics

In [5]:
# ¿Cuáles son los 3 productos más vendidos (en cantidad) según el tipo de pedido?
# Ranking de Productos usando Window Functions

query_top_items = """
    WITH ventas_por_tipo_pedido AS (
        SELECT
            location as tipo_pedido,
            item as producto,
            SUM(quantity) as total_vendido
        FROM df_silver 
        GROUP BY tipo_pedido, producto
    ),
    ranking_productos AS (
        SELECT 
            tipo_pedido,
            producto,
            total_vendido,
            RANK() OVER(
                PARTITION BY tipo_pedido ORDER BY total_vendido DESC
            ) as ranking_ventas
        FROM ventas_por_tipo_pedido
    )
    
    SELECT *
    FROM ranking_productos
    WHERE ranking_ventas <= 3
    ORDER BY tipo_pedido, ranking_ventas;
"""

df_top_3_items : pd.DataFrame = duckdb.query(query_top_items).df()

print("--- Top 3 Productos Más Vendidos por Tipo de Pedido ---")
print(df_top_3_items)

--- Top 3 Productos Más Vendidos por Tipo de Pedido ---
  tipo_pedido  producto  total_vendido  ranking_ventas
0    In-Store  Sandwich         1021.0               1
1    In-Store     Juice         1020.0               2
2    In-Store     Salad         1011.0               3
3    Takeaway    Cookie         1008.0               1
4    Takeaway    Coffee          988.0               2
5    Takeaway     Salad          959.0               3
6     Unknown    Coffee         1375.0               1
7     Unknown  Smoothie         1339.0               2
8     Unknown     Juice         1302.0               3


Descubrir si los productos (ej. "Coffee") tienen siempre el mismo precio o si hay variaciones (y cuán grandes son)

In [6]:
query_coffee_price = """
    SELECT 
        item AS producto,
        MIN(price_per_unit) AS precio_min,
        MAX(price_per_unit) AS precio_max,
        COUNT(DISTINCT price_per_unit) AS precios_diferentes
    FROM df_silver
    GROUP BY item
    ORDER BY item DESC
"""

df_coffee_price : pd.DataFrame = duckdb.query(query_coffee_price).df()

df_coffee_price
# Se mantuvo el mismo precio todo el año

,producto,precio_min,precio_max,precios_diferentes
0,Tea,1.5,1.5,1
1,Smoothie,4.0,4.0,1
2,Sandwich,4.0,4.0,1
3,Salad,5.0,5.0,1
4,Juice,3.0,3.0,1
5,Cookie,1.0,1.0,1
6,Coffee,2.0,2.0,1
7,Cake,3.0,3.0,1
8,NaN,1.0,5.0,6


Encontrar transacciones con cantidades (quantity) sospechosamente altas para una cafetería.

In [ ]:
query_quantity = """
    WITH metricas_por_producto AS (
        SELECT 
            transaction_id,
            item AS producto,
            quantity AS cantidad_comprada,
            -- Calculamos el promedio global de ventas de ESE producto en todo el año
            AVG(quantity) OVER(PARTITION BY item) AS promedio_global_producto
        FROM df_silver
    )
    SELECT *
    FROM metricas_por_producto
    -- Filtramos compras que sean, por ejemplo, 3 veces mayores al promedio normal
    WHERE cantidad_comprada > (promedio_global_producto * 3)
    ORDER BY cantidad_comprada DESC
    LIMIT 10;
"""

df_quantity : pd.DataFrame = duckdb.query(query_quantity).df()

df_quantity
# NO hay transacciones concantidades sospechosas

,transaction_id,producto,cantidad_comprada,promedio_global_producto
0,TXN_3578141,Cake,5,3.030888
1,TXN_2996519,Cake,5,3.030888
2,TXN_1435086,Cake,5,3.030888
3,TXN_8876618,Cake,5,3.030888
4,TXN_8377564,Cake,5,3.030888
5,TXN_2368126,Cake,5,3.030888
6,TXN_8982764,Cake,5,3.030888
7,TXN_7742742,Cake,5,3.030888
8,TXN_8718498,Cake,5,3.030888
9,TXN_8703771,Cake,5,3.030888


Validar si existen ventas registradas para los 365 días del año 2023, o si la cafetería "cerró" algún día.

In [8]:
query_date = """
    SELECT
        COUNT(DISTINCT transaction_date) AS dias_activos_en_2023
    FROM df_silver
    ORDER BY dias_activos_en_2023 DESC
"""

df_date : pd.Series = duckdb.query(query_date).df()

df_date
# La tienda estuvo activa todo el año

,dias_activos_en_2023
0,365


In [ ]:
query_date = """
    SELECT
        -- CAST trunca la hora y se queda solo con AAAA-MM-DD
        COUNT(DISTINCT CAST(transaction_date AS DATE)) AS dias_activos_en_2023
    FROM df_silver
"""

df_date : pd.Series = duckdb.query(query_date).df()

df_date